# F1-scientific-python — Session 03: Random Numbers and Plotting

**Session length:** about 75 minutes • **Concepts:** random-seeding,
matplotlib-basics — plus the unit's exam connections and where these skills go
next.

Checkpoint answers are collected at the end of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Random generators

You will constantly need made-up-but-realistic numbers: test data for a
function, dice rolls, noisy measurements for a plot. NumPy's random generator
provides them.

The modern interface is `np.random.default_rng(seed)`, which returns a
**generator** object, usually named `rng`. Its most useful methods:

- `rng.random(size)` — decimals from 0.0 up to (but not including) 1.0
- `rng.integers(low, high, size)` — integers from `low` up to (excluding) `high`
- `rng.choice(arr, size)` — elements picked from an existing array
- `rng.shuffle(arr)` — mix an array's order, **in place**

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

print("decimals:", rng.random(3))
print("dice:    ", rng.integers(1, 7, size=10))      # 7 excluded -> faces 1..6
print("choice:  ", rng.choice(np.array(["red", "green", "blue"]), size=5))

deck = np.arange(10)
rng.shuffle(deck)
print("shuffled:", deck)

### Checkpoint 1

1. Build a generator seeded with 42 and draw 5 integers from 0 to 9.
2. `rng.integers(1, 7, size=10)` never produces a 7. Why not, and how do you
   get faces 1 through 7 inclusive?

## 2. Seeds and the reproducibility discipline

**What a seed is.** The "random" numbers are actually produced by a completely
deterministic recipe; the **seed** is the recipe's starting point. Same seed →
exactly the same sequence of numbers, every run, on every machine. No seed →
NumPy grabs an unpredictable starting point, and every run differs.

In [ ]:
rng_a = np.random.default_rng(123)
rng_b = np.random.default_rng(123)
rng_c = np.random.default_rng(999)

print("same seed, same numbers?     ", np.array_equal(rng_a.random(5), rng_b.random(5)))
print("different seed, same numbers?", np.array_equal(np.random.default_rng(123).random(5),
                                                      rng_c.random(5)))

**Why seeding is a discipline, not a detail.** If your notebook uses random
numbers *without* a seed, nobody — including you tomorrow — can rerun it and get
the same results. Bugs become impossible to reproduce, and graders cannot verify
your numbers. So follow this rule in every notebook you write in this course:

1. Define a single `SEED` constant at the top.
2. Create one generator: `rng = np.random.default_rng(SEED)`.
3. Draw every random number from that `rng`.

Note that the *order* of draws matters: one `rng` hands out a single fixed
sequence, so adding an extra draw shifts everything after it. Rerun from the
top when you change random code.

### Checkpoint 2

1. Why does re-running a cell that builds a seeded generator and draws from it
   print the same numbers every time?
2. What changes if you write `np.random.default_rng()` with no seed, and why is
   that a problem for graded work?

## 3. A seeded experiment, end to end

Let's put the discipline to work — roll 10,000 dice and see how often a 6 comes
up (it should be close to 1/6 ≈ 0.1667). Note the vectorized counting trick
from Session 02: a mask's `.mean()` is the fraction of True values.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
rolls = rng.integers(1, 7, size=10_000)
print("fraction of sixes:", (rolls == 6).mean())

Anyone who runs this notebook gets *exactly* that fraction — same seed, same
rolls, same answer. That is what makes the number checkable.

### Checkpoint 3

1. With the same seeded setup, what fraction of the 10,000 rolls are 5 or
   higher? Answer with one loop-free expression.
2. Roll two dice 10,000 times (two separate `integers` draws) and compute the
   fraction of rolls whose total is exactly 7.

## 4. Line plots

Numbers in bulk are hard to read; a picture is instant. matplotlib is the
standard Python plotting library. A **line plot** (`plt.plot`) shows how one
quantity changes along another; points are connected in order. Use it for
functions and anything measured over time.

Every plot you make must be labeled — `plt.title`, `plt.xlabel`, `plt.ylabel`,
and `plt.legend()` when two or more things share the axes (each gets a
`label=`). An unlabeled plot is an unanswered question. Finish each figure with
`plt.show()`.

In [ ]:
x = np.linspace(-1, 5, 100)

plt.figure(figsize=(6, 4))
plt.plot(x, x**2 - 4*x + 3, label="y = x^2 - 4x + 3")
plt.plot(x, 2*x - 3, label="y = 2x - 3", linestyle="--")
plt.title("A curve and a line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### Checkpoint 4

1. Plot `y = 3x + 1` for 50 x-values from 0 to 5, with a title and both axis
   labels.
2. Why does `plt.legend()` show nothing unless each `plot` call got a
   `label=`?

## 5. Scatter plots

A **scatter plot** (`plt.scatter`) draws individual (x, y) points,
unconnected. Use it to see whether two measured quantities move together —
each point is one observation, and there is no meaningful order connecting one
point to the next.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
study_hours = rng.random(40) * 10                    # 40 values in [0, 10)
quiz_score = 40 + 5 * study_hours + rng.integers(-8, 9, size=40)

plt.figure(figsize=(6, 4))
plt.scatter(study_hours, quiz_score)
plt.title("Quiz score vs. hours studied")
plt.xlabel("hours studied")
plt.ylabel("quiz score")
plt.show()

### Checkpoint 5

1. In one sentence: when do you reach for `scatter` instead of `plot`?
2. What would this figure look like if you had used `plt.plot` by mistake, and
   why?

## 6. Histograms

A **histogram** (`plt.hist`) takes ONE array of values, chops its range into
`bins` intervals, and draws how many values land in each. Use it to see where
values pile up.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
rolls = rng.integers(1, 7, size=500) + rng.integers(1, 7, size=500)   # 500 two-dice totals

plt.figure(figsize=(6, 4))
plt.hist(rolls, bins=11)          # totals run 2..12 -> 11 bins
plt.title("Totals of 500 two-dice rolls")
plt.xlabel("total")
plt.ylabel("how many rolls")
plt.show()

Totals near 7 tower over 2 and 12 — there are simply more ways to make 7. The
histogram shows that at a glance; a printout of 500 numbers never would.
(`np.histogram(values, bins)` returns the same bar heights and bin edges as
numbers, which is how you *check* a histogram in code.)

### Checkpoint 6

1. Roll 500 single dice with a seeded generator and draw a histogram with
   6 bins.
2. Too few bins hide detail; too many make noise. What does `bins=2` show about
   the two-dice totals above? Try it.

## 7. The figure-quality checklist

On every figure you produce in this course (and in practice problems, where
solutions are checked):

1. **Title** says what the figure shows.
2. **Both axis labels** name the quantities (units where relevant).
3. **Legend** whenever two or more things share the axes.
4. **Right plot type**: connected order → line; separate observations →
   scatter; one array's value spread → histogram.
5. **Seeded data** if anything random is drawn.

### Checkpoint 7

1. This snippet draws a figure that breaks the checklist:
   `plt.plot(np.arange(5), np.arange(5) ** 2); plt.show()` — list every
   violation and fix them.
2. You measured one array of 300 reaction times and want to show where they
   cluster. Which plot type, and why?

## 8. Exam Connections

Everything in this unit is directly examinable. Round 1's topic mix
includes a cluster of scientific-Python/NumPy problems (see
`reference/analysis.md`), and they arrive in a few recognizable shapes — all
paraphrased here, none quoted:

- **Exact-contract coding tasks.** "Implement exactly `def f(data):` ...";
  the function name, argument order, and the result's shape are specified, and
  graders test against that contract literally. A right idea under a wrong name
  or shape scores nothing.
- **API bans with a zero-points clause.** "Your solution must not contain
  `for` or `while`" (sometimes specific functions are banned too). Submissions
  are checked mechanically — this is why Session 02's vectorization drills
  exist.
- **Predict-the-output multiple choice.** Five options A–E; you trace shapes,
  broadcasting, and indexing by hand, no computer. Numeric answers may be
  required in a normal form (e.g., a fraction in lowest terms).
- **Reproducibility requirements.** Tasks that involve generated numbers fix a
  seed so every contestant and grader sees identical data — the Section 2
  discipline, verbatim.

**Worked exam-style example: predict the output.** Solved by hand, step by
step:

> Let `M = np.arange(12).reshape(3, 4)`. What is the value of
> `M[:, 1:].sum(axis=0)[0]`?
>
> (A) 12  (B) 15  (C) 3  (D) 18  (E) 21

1. `M` holds 0–11 in 3 rows of 4: rows `[0 1 2 3]`, `[4 5 6 7]`, `[8 9 10 11]`.
2. `M[:, 1:]` keeps all rows, drops column 0: rows `[1 2 3]`, `[5 6 7]`,
   `[9 10 11]` — shape `(3, 3)`.
3. `.sum(axis=0)`: axis 0 disappears, one total per remaining column:
   `[1+5+9, 2+6+10, 3+7+11] = [15, 18, 21]`.
4. `[0]` picks the first: **15 → answer (B)**.

Verify (allowed here, not in the exam room):

In [ ]:
M = np.arange(12).reshape(3, 4)
print(M[:, 1:].sum(axis=0)[0])

### Checkpoint 8

1. Exam-style, by hand first: `np.arange(10)[2:8:2].mean()` — (A) 3.0
   (B) 4.0 (C) 5.0 (D) 4.5 (E) an error. Show your steps.
2. A task asks for an answer "as a fraction p/q in lowest terms with q > 0".
   Your computation gives 6/8. What exactly do you submit?

## 9. Going Deeper

Nothing below is needed for this unit's practice set — it is a map of where
these skills go next along the course:

- **F2-vectors** (the next unit) picks up exactly where this one stops: the
  1-D arrays you built here become measurable geometric objects, and every new
  operation there is assembled from this unit's parts — elementwise products,
  sums, square roots, broadcasting. If you can write `(a * b).sum()` without
  thinking, F2 will feel like naming things you already do.
- **F6-svd-spectral** (much later) works with 2-D arrays as objects that can be
  taken apart into simpler pieces and rebuilt. The habits that matter there are
  the ones drilled here: reading shapes, tracking which axis is which, and
  trusting whole-array operations over loops.

If you want more practice *now*, the highest-value drill is self-made: take any
loop you have ever written over a list of numbers and rewrite it loop-free,
then verify with `np.allclose`.

### Checkpoint 9

1. Warm up for F2-vectors without leaving this unit: for
   `a = np.array([1.0, 2.0, 2.0])` and `b = np.array([3.0, 0.0, 4.0])`,
   compute `(a * b).sum()` and `np.sqrt((a ** 2).sum())` loop-free.
2. Pick any loop you wrote before this course and rewrite it loop-free; verify
   with `np.allclose` against the original.

## Checkpoint Answers

### Checkpoint 1 answers

In [ ]:
rng = np.random.default_rng(42)          # 1.
print(rng.integers(0, 10, size=5))

# 2. integers(low, high) EXCLUDES high. For faces 1..7 use rng.integers(1, 8, ...).

### Checkpoint 2 answers

In [ ]:
# 1. Building the generator from the seed restarts the deterministic recipe at
#    the same point, so the sequence of draws is identical every run.
# 2. With no seed, the starting point is unpredictable, so every run produces
#    different numbers — results are not reproducible and graders cannot verify
#    them.
print("see comments")

### Checkpoint 3 answers

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
rolls = rng.integers(1, 7, size=10_000)
print("5 or higher:", (rolls >= 5).mean())            # 1.

rng = np.random.default_rng(SEED)                     # 2.
d1 = rng.integers(1, 7, size=10_000)
d2 = rng.integers(1, 7, size=10_000)
print("total is 7:", (d1 + d2 == 7).mean())           # near 6/36 = 0.1667

### Checkpoint 4 answers

In [ ]:
x = np.linspace(0, 5, 50)                             # 1.
plt.figure(figsize=(6, 4))
plt.plot(x, 3 * x + 1)
plt.title("y = 3x + 1")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# 2. The legend is built FROM the label= entries; with no labels there is
#    nothing for it to list.

### Checkpoint 5 answers

In [ ]:
# 1. Use scatter when the data are separate (x, y) observations with no
#    meaningful order connecting one point to the next.
# 2. plt.plot would connect the points in array order, drawing meaningless
#    zig-zags across the cloud — the order of students carries no information.
print("see comments")

### Checkpoint 6 answers

In [ ]:
rng = np.random.default_rng(20260804)                 # 1.
rolls = rng.integers(1, 7, size=500)
plt.figure(figsize=(6, 4))
plt.hist(rolls, bins=6)
plt.title("500 single dice rolls")
plt.xlabel("face")
plt.ylabel("how many rolls")
plt.show()

# 2. bins=2 lumps totals 2-7 and 7-12 into two bars: you only learn that the
#    halves are roughly balanced — the peak at 7 vanishes.
totals = rng.integers(1, 7, size=500) + rng.integers(1, 7, size=500)
plt.figure(figsize=(6, 4))
plt.hist(totals, bins=2)
plt.title("Two-dice totals, bins=2 (too coarse)")
plt.xlabel("total")
plt.ylabel("how many rolls")
plt.show()

### Checkpoint 7 answers

In [ ]:
# 1. Violations: no title, no x label, no y label. Fixed:
plt.figure(figsize=(6, 4))
plt.plot(np.arange(5), np.arange(5) ** 2)
plt.title("y = x^2 at integer x")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# 2. A histogram — one array, and the question is where its values pile up.

### Checkpoint 8 answers

In [ ]:
# 1. Step 1: np.arange(10)[2:8:2] keeps positions 2, 4, 6 -> [2, 4, 6].
#    Step 2: mean = 12/3 = 4.0 -> answer (B).
print(np.arange(10)[2:8:2].mean())

# 2. Reduce 6/8 by gcd 2 -> submit 3/4 (q = 4 > 0 already).

### Checkpoint 9 answers

In [ ]:
a = np.array([1.0, 2.0, 2.0])
b = np.array([3.0, 0.0, 4.0])
print((a * b).sum())                  # 1. 3 + 0 + 8 = 11
print(np.sqrt((a ** 2).sum()))        #    sqrt(1 + 4 + 4) = 3.0

# 2. Personal exercise — the pattern: run both versions, then np.allclose.